# 07a: Lags + stepwise selection

A rebuild of `scripts/06_model_selection/` (same data, same random 80/20 split by whole days, seed 860, same
50 busiest zones, same `log(pickups + 1)` target and day-clustered standard errors), as **one notebook** in a
new order. This version keeps **yesterday's and last week's demand** (the lags) as predictors and picks the model with **stepwise selection by BIC**.

| Version | Lags (yesterday, last week) | Model selection |
|---|---|---|
| `07a_lags_stepwise.ipynb` | yes | stepwise (BIC) |
| `07b_no_lags_stepwise.ipynb` | no | stepwise (BIC) |
| `07c_no_lags_holistic.ipynb` | no | holistic (adjusted R², AIC, BIC, explainability) |

**What changed from 06 (all three versions).** Month, day of week, holiday and the weekday/weekend hour
pattern are no longer dummy variables in the model. They are stripped out first with **Frisch-Waugh-Lovell
(FWL)**, and every later step (regression, residual and QQ plots, F-tests and collinearity, model selection,
forecast accuracy) works on the deseasonalized data. The FWL theorem guarantees that the slopes are the same
as with the dummies, so this replaces the dummies rather than checking them.

**Order of the sections:**

1. Create the lags (yesterday and last week)
2. Split into training and test days (random, seed 860)
3. Deseasonalize with FWL on the training days
4. The regression
5. Residual plot
6. QQ plot
7. F-tests and collinearity
8. Model selection (stepwise by BIC)
9. Forecast accuracy (RMSE and MAE on the test days)

**Needs:** `data/processed/model_table.parquet` (run `scripts/01_` to `04_` first). Writes nothing: results
are shown in the notebook. Runs in about a minute.

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import patsy
import statsmodels.api as sm
from pandas.tseries.holiday import USFederalHolidayCalendar
from statsmodels.tools.eval_measures import meanabs, rmse

# The repo root is the first folder above this notebook that has data/processed/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "processed").is_dir())
MODEL_TABLE = ROOT / "data" / "processed" / "model_table.parquet"

SEED = 860            # random test days (same seed as 06_model_selection)
TEST_SHARE = 0.2      # 80/20 split, by whole days
TOP_ZONES = 50        # busiest zones by training pickups
ALPHA = 0.05          # significance level
AUX_R2_FLAG = 0.8     # a predictor more than 80% explained by the other predictors is flagged as collinear
PAIR_CORR_FLAG = 0.7  # pairs with a correlation above this are tested together

# Regular predictors (everything that is not part of the seasonal pattern Z, see Section "Deseasonalize").
RAW_WEATHER = ["temperature_2m", "precipitation", "snowfall", "snow_depth", "cloud_cover",
               "shortwave_radiation", "is_day"]
WEATHER = ["temperature_2m", "temp_sq", "precipitation", "snowfall", "snow_depth", "cloud_cover",
           "shortwave_radiation", "is_day"]
LAGS = ["log_lag24", "log_lag168"]  # log(pickups + 1) in the same zone and clock hour 1 and 7 days earlier
CONTINUOUS = WEATHER + LAGS
ALL_TERMS = ["zone", "zone_x_weekend"] + CONTINUOUS
PARENTS = {"zone_x_weekend": ["zone"], "temp_sq": ["temperature_2m"]}  # a term needs the terms it is built from

INK, INK_2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": GRID, "axes.labelcolor": INK_2,
    "text.color": INK, "xtick.color": INK_2, "ytick.color": INK_2, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left", "figure.dpi": 110,
})
pd.set_option("display.width", 160, "display.max_columns", 30, "display.float_format", "{:,.4f}".format)

con = duckdb.connect()
con.execute(f"CREATE VIEW m AS SELECT * FROM '{MODEL_TABLE.as_posix()}'")

## 1. Create the lags (yesterday and last week)

For every zone-hour, the pickups in the **same zone at the same clock hour** 1 day earlier (`lag24`) and
7 days earlier (`lag168`). They are built from the whole table (all 263 zones, every day) before the
split, so a training day can use a test day's demand as its "yesterday", exactly as in 06. Both enter the
model in logs (`log(lag + 1)`), like the target.

A lag is missing in the first 1 or 7 days of January, and next to the skipped daylight-saving hour
(2025-03-09 02:00). Those rows are dropped in the next section.

In [ ]:
df = con.execute(f"""
    SELECT m.zone, m.hour, m.pickups, {", ".join("m." + c for c in RAW_WEATHER)},
           l24.pickups AS lag24, l168.pickups AS lag168
    FROM m
    LEFT JOIN m l24  ON l24.zone  = m.zone AND l24.hour  = m.hour - INTERVAL 24 HOUR
    LEFT JOIN m l168 ON l168.zone = m.zone AND l168.hour = m.hour - INTERVAL 168 HOUR
    ORDER BY m.zone, m.hour
""").df()
df = df.astype({c: "float64" for c in df.columns if str(df[c].dtype) in ("Float64", "Int64", "Int32")})
df["log_lag24"] = np.log1p(df["lag24"])
df["log_lag168"] = np.log1p(df["lag168"])
print(f"{len(df):,} zone-hours (all 263 zones)")
df[["lag24", "lag168"]].isna().sum().rename("missing").to_frame()

## 2. Split into training and test days

Same split as 06: **20% of the 365 days are drawn at random as test days** (seed 860), the other 80% are
training days. Whole days go to one side, so the hours of a test day are never seen in training. Every month,
weekday and hour is in the training data, so every seasonal coefficient can be estimated.

The model uses the **50 busiest zones**, ranked by pickups on the training days. Rows with missing values are
dropped: the last hour of the year (no weather after the end-of-hour shift), and the rows without a lag (first week of January, hours next to the skipped DST hour).

In [ ]:
df["snow_depth"] = df["snow_depth"] * 100   # metres -> cm, so its coefficient reads per cm like snowfall
df["date"] = df["hour"].dt.normalize()
days = pd.date_range("2025-01-01", "2025-12-31", freq="D")
test_days = pd.DatetimeIndex(np.random.default_rng(SEED).choice(
    days, size=round(TEST_SHARE * len(days)), replace=False)).sort_values()
df["split"] = np.where(df["date"].isin(test_days), "test", "train")

top = df[df["split"] == "train"].groupby("zone")["pickups"].sum().nlargest(TOP_ZONES).index
df = df[df["zone"].isin(top)].dropna().reset_index(drop=True)

# Calendar columns: used only to build the seasonal pattern Z in the next section.
holidays = USFederalHolidayCalendar().holidays("2025-01-01", "2025-12-31")
df["hour_of_day"] = df["hour"].dt.hour
df["dow"] = df["hour"].dt.dayofweek + 1                      # 1 = Monday ... 7 = Sunday
df["month"] = df["hour"].dt.month
df["day"] = df["hour"].dt.dayofyear                          # day id for day-clustered standard errors
df["holiday"] = df["date"].isin(holidays).astype(float)
df["weekend"] = (df["dow"] >= 6).astype(float)
df["hour_wknd"] = np.where(df["weekend"] == 1, df["hour_of_day"], 0)  # weekend hour, 0 on weekdays
df["y"] = np.log1p(df["pickups"])                            # the target: log(pickups + 1)

train = df[df["split"] == "train"].reset_index(drop=True)
test = df[df["split"] == "test"].reset_index(drop=True)
temp_mean = train["temperature_2m"].mean()                  # temp_sq is centred on the training mean
for d in (train, test):
    d["temp_sq"] = (d["temperature_2m"] - temp_mean) ** 2
del df

print(f"{len(test_days)} test days, {len(days) - len(test_days)} training days, {len(top)} zones")
print(f"train {len(train):,} rows, test {len(test):,} rows ({len(test) / (len(train) + len(test)):.0%} test)")
for name, d in [("training", train), ("test", test)]:
    hol = d.loc[d["holiday"] == 1, "date"].drop_duplicates().dt.strftime("%b %d")
    print(f"holidays in the {name} days ({len(hol)}): {', '.join(hol)}")

## 3. Deseasonalize with Frisch-Waugh-Lovell (training days only)

**Z, the seasonal pattern that is stripped out** (65 columns):
- the constant (intercept),
- day of week (6 dummies, Monday is the reference),
- month (11 dummies, January is the reference),
- holiday (1 dummy, US federal holidays),
- the hour-of-day pattern **separately for weekdays and weekends** (23 hour dummies for the weekday pattern
  + 23 weekend-hour dummies for how weekends differ: the weekend × hour effect).

**The FWL steps, fitted on the training days only:**
1. Regress **Y** = `log(pickups + 1)` on Z and keep the residuals **Ỹ**: demand that is higher or lower than
   usual for that hour, weekday, month and holiday.
2. Regress **each regular predictor** on Z and keep the residuals **X̃**. For example, deseasonalized
   temperature is "how much warmer or colder than usual for this hour and month".
3. From here on, every regression is **Ỹ on X̃**. By the FWL theorem its slopes are exactly the slopes of the
   regression with all the Z dummies included, so the dummies are not needed in the model.

**Test days** are deseasonalized with the coefficients from the training days (nothing is fitted on test
days). A prediction for a test hour = its seasonal part (from Z) + the regular predictors' part.

**The regular predictors** are everything outside Z: zone (49 dummies, the busiest zone is the reference),
zone × weekend (49 dummies: how each zone's weekend differs), the weather and the two lags. Zone isn't a time pattern, so it isn't in Z; it is deseasonalized like
the other predictors (this barely changes it, because every zone has almost every hour of the year).

**Parameters still count.** Z's 65 coefficients are estimated in step 1, so they are counted in adjusted R²,
AIC and BIC. R² is shown two ways: **total** (share of `log(pickups + 1)` explained, with Z) and **after Z**
(share of the deseasonalized demand Ỹ explained by the regular predictors).

In [ ]:
Z_FORMULA = "C(dow) + C(month) + holiday + C(hour_of_day) + C(hour_wknd)"
Z_tr = patsy.dmatrix(Z_FORMULA, train, return_type="dataframe")
Z_te = patsy.build_design_matrices([Z_tr.design_info], test, return_type="dataframe")[0]
K_Z = Z_tr.shape[1]

ref_zone = int(train.groupby("zone")["pickups"].sum().idxmax())
zones = sorted(int(z) for z in top if z != ref_zone)
TERMS = {"zone": [f"zone[{z}]" for z in zones],                       # term -> its columns
         "zone_x_weekend": [f"zone_x_weekend[{z}]" for z in zones],
         **{c: [c] for c in CONTINUOUS}}


def regular_predictors(d):
    cols = {f"zone[{z}]": (d["zone"] == z).astype(float) for z in zones}
    cols |= {f"zone_x_weekend[{z}]": ((d["zone"] == z) & (d["weekend"] == 1)).astype(float) for z in zones}
    cols |= {c: d[c].astype(float) for c in CONTINUOUS}
    return pd.DataFrame(cols)


def deseasonalize(cols_tr, cols_te):
    """FWL: regress every column on Z (training days) and keep the residuals. Test days use the same
    training-day coefficients. Returns (train residuals, test residuals, test seasonal part)."""
    coef = np.linalg.lstsq(Z_tr.to_numpy(), cols_tr.to_numpy(), rcond=None)[0]
    seasonal_tr = pd.DataFrame(Z_tr.to_numpy() @ coef, columns=cols_tr.columns)
    seasonal_te = pd.DataFrame(Z_te.to_numpy() @ coef, columns=cols_te.columns)
    return cols_tr - seasonal_tr, cols_te - seasonal_te, seasonal_te


# Step 1: the target (and raw pickups, used only to justify the log in the residual plot).
targets_tr, targets_te = train[["y", "pickups"]].astype(float), test[["y", "pickups"]].astype(float)
Yt_tr, Yt_te, Yseas_te = deseasonalize(targets_tr, targets_te)
# Step 2: every regular predictor.
Xt_tr, Xt_te, _ = deseasonalize(regular_predictors(train), regular_predictors(test))

seasonal_share = pd.Series({c: 1 - Xt_tr[c].var() / train[c].var() for c in CONTINUOUS})
seasonal_share = pd.concat([pd.Series({"log(pickups + 1)": 1 - Yt_tr["y"].var() / targets_tr["y"].var()}),
                            seasonal_share])
print(f"Z: {K_Z} columns; regular predictors: {Xt_tr.shape[1]} columns "
      f"({len(zones)} zone, {len(zones)} zone x weekend, {len(CONTINUOUS)} numeric)")
seasonal_share.rename("share explained by the seasonal pattern Z (training days)").to_frame()

**Results.**

- The seasonal pattern Z explains **56%** of the variation in `log(pickups + 1)`. Most of what is left (Ỹ) is the
  difference between zones, which the regular predictors pick up next.
- **Temperature (86%), `is_day` (82%) and sunlight (76%) are mostly seasonal**: knowing the hour and the month already
  tells you most of their value. After FWL their effects are estimated from what is left (e.g. an afternoon that is
  warmer than usual for June), so their confidence intervals are wider.
- **Rain (2%), snowfall (3%) and cloud cover (8%) are hardly seasonal at all**: almost all of their variation is
  "unusual weather", which is exactly what is needed to measure their effect.
- **The lags are about half seasonal** (52% and 55%): part of yesterday's and last week's demand is just the same
  hour-and-weekday pattern repeating.

## 4. The regression

Ỹ regressed on every regular predictor's X̃ (no intercept needed: Z already contains it, so every residual has
mean 0). Standard errors are **clustered by day**, because hours on the same day share the weather and are
not independent.

Because the target is in logs, a coefficient converts to a **% change in pickups**: `100 × (e^coef − 1)`.
For the lags, read the coefficient as an elasticity: 10% more pickups last week goes with about (10 × coef)% more now. `temp_sq`
is the squared distance from the average training temperature (printed below), so
`temperature_2m` is the effect of 1 °C at an average temperature. The 98 zone and zone × weekend coefficients
are not printed: they are tested as groups in the F-test section.

In [ ]:
N = len(train)
SST = {t: ((targets_tr[t] - targets_tr[t].mean()) ** 2).sum() for t in targets_tr}


def fit(terms, target="y"):
    """Ỹ on the X̃ of these terms, standard errors clustered by day."""
    cols = [c for t in terms for c in TERMS[t]]
    return sm.OLS(Yt_tr[target], Xt_tr[cols]).fit(cov_type="cluster", cov_kwds={"groups": train["day"]},
                                                  use_t=True)


def fit_stats(ssr, k_regular, target="y"):
    """Fit measures of the whole model (Z + regular predictors) from its sum of squared residuals."""
    k = k_regular + K_Z                                     # Z's coefficients are parameters too
    llf = -N / 2 * (np.log(2 * np.pi) + np.log(ssr / N) + 1)
    return {"parameters": k, "R2": 1 - ssr / SST[target],
            "R2_adj": 1 - (ssr / (N - k)) / (SST[target] / (N - 1)),
            "R2 after Z": 1 - ssr / (Yt_tr[target] ** 2).sum(),
            "AIC": -2 * llf + 2 * k, "BIC": -2 * llf + np.log(N) * k}


res_all = fit(ALL_TERMS)
print(f"average training temperature: {temp_mean:.1f} °C")
print(pd.Series(fit_stats(res_all.ssr, len(res_all.params))).to_string(float_format="{:,.4f}".format))

UNIT = {"temperature_2m": "per °C", "temp_sq": "per °C² from the mean", "precipitation": "per mm",
        "snowfall": "per cm", "snow_depth": "per cm on the ground", "cloud_cover": "per % point",
        "shortwave_radiation": "per W/m²", "is_day": "daylight vs dark", "log_lag24": "elasticity", "log_lag168": "elasticity"}
ci = res_all.conf_int()
coefs = pd.DataFrame({"unit": pd.Series(UNIT), "coef": res_all.params, "std err": res_all.bse,
                      "p": res_all.pvalues, "95% CI low": ci[0], "95% CI high": ci[1]}).loc[CONTINUOUS]
coefs["% change in pickups"] = 100 * np.expm1(coefs["coef"])
coefs

Note: the FWL regression doesn't know that Z used 65 degrees of freedom, so its clustered standard errors are
a tiny bit too small. The correction factor is printed below; it is far too small to change any conclusion,
so it is left out.

In [ ]:
k = len(res_all.params)
print(f"standard errors would grow by {100 * (np.sqrt((N - k) / (N - k - K_Z)) - 1):.4f}%")

**Results.**

The whole model (Z + regular predictors) has **173 parameters**: R² = 0.916, adjusted R² = 0.916, AIC = 311,793,
BIC = 313,654. After Z, the regular predictors explain **81%** of the deseasonalized demand.

- **Rain** (`precipitation`, coef 0.021, p < 0.001): each mm of rain in the hour raises pickups by about **2.1%**.
- **Last week's and yesterday's pickups** (`log_lag168` 0.564, `log_lag24` 0.268, both p < 0.001): 10% more pickups at
  the same zone and hour last week goes with about 5.6% more now; 10% more yesterday with about 2.7% more.
- **Sunlight** (p = 0.03): slightly *fewer* pickups when it is brighter than usual for the time: about −0.8% per
  100 W/m².
- **Not significant:** temperature and temperature² (p = 0.54 and 0.60), `is_day` (0.14), cloud cover (0.16), snow on
  the ground (0.89) and snowfall (0.91). Their confidence intervals include 0.
- Rain and the two lags match 06 (0.021, 0.268, 0.564) to three decimals: FWL gives the same slopes as the dummies.

## 5. Residual plot

The tests assume the errors (actual − predicted) have **the same spread** at every demand level. The residuals
of the FWL regression are exactly the residuals of the full model, and "fitted" is the full prediction
(seasonal part + regular predictors). An even band around 0 is good; a funnel means the errors grow with
demand. The same model on **raw pickups** is shown next to it, to check that the log target is still the
right choice.

In [ ]:
res_raw = fit(ALL_TERMS, target="pickups")
models = {"Raw pickups": (targets_tr["pickups"] - res_raw.resid, res_raw.resid),
          "log(pickups + 1)": (targets_tr["y"] - res_all.resid, res_all.resid)}
print(f"raw pickups: {(models['Raw pickups'][0] < 0).mean():.2%} of fitted values are negative; "
      f"log: {(np.expm1(models['log(pickups + 1)'][0]) < 0).mean():.2%}")
del res_raw

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, (label, (fitted, resid)) in zip(axes, models.items()):
    s = fitted.sample(20_000, random_state=SEED).index
    ax.scatter(fitted[s], resid[s], s=4, alpha=0.25, color=BLUE, linewidths=0)
    ax.axhline(0, color=INK_2, lw=1)
    ax.set(title=f"{label}: residuals vs fitted", xlabel="fitted", ylabel="residual")
fig.tight_layout()
plt.show()

**Results.**

- **Raw pickups:** a funnel (the busier the hour, the bigger the errors), and **13%** of fitted values are negative
  pickups.
- **`log(pickups + 1)`:** a much more even band around 0, and only 0.01% of fitted values are negative. The diagonal
  lines on the left are zone-hours with 0, 1, 2, ... pickups: counts are whole numbers, so their residuals fall on lines.

**Decision:** keep `log(pickups + 1)`, as in 06. The uneven spread that is left is handled by the day-clustered standard
errors used in every test.

## 6. QQ plot

Checks whether the errors are roughly **normally distributed**. Points on the dashed line mean normal errors;
points bending away at the ends mean more extreme errors than a normal distribution has. Both axes are in
standard deviations.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, (label, (fitted, resid)) in zip(axes, models.items()):
    sm.qqplot(resid.sample(5_000, random_state=SEED), fit=True, line="45", ax=ax,
              marker=".", markersize=3, alpha=0.5, markerfacecolor=BLUE, markeredgecolor=BLUE)
    ax.get_lines()[1].set(color=INK_2, linewidth=1.2, linestyle="--")
    x, y = ax.get_lines()[0].get_xdata(), ax.get_lines()[0].get_ydata()
    ax.set(xlim=(x.min() - 0.3, x.max() + 0.3), ylim=(y.min() - 0.3, y.max() + 0.3))
    ax.set(title=f"{label}: normal QQ plot", xlabel="theoretical quantile", ylabel="standardised residual")
fig.tight_layout()
plt.show()
del models

**Results.**

- **Raw pickups:** the points bend far above the line on the right: many more very large errors than a normal
  distribution has.
- **`log(pickups + 1)`:** close to the line between about −2 and +2 standard deviations. The left end bends far down: a
  few hours have much less demand than predicted (holidays, storms, events). With 345,000 rows the tests don't depend on
  normal errors, so this is noted, not fixed.

## 7. F-tests and collinearity

**F-test per term:** "are all of this term's coefficients zero?" A group of dummies (the 49 zones, the 49
zone × weekend shifts) is tested as a whole. Clustered by day. **Non-significant terms are reported but not
removed for that reason alone**: "not significant" means the data can't measure the effect precisely, not
that it is zero.

**Collinearity** (predictors that carry the same information, so their effects can't be told apart), checked
with auxiliary regressions and F-tests (no VIF):
1. **Share explained by Z.** How seasonal a predictor is. This is no longer a collinearity problem: FWL has
   already removed the seasonal part, and the coefficient is estimated from what is left (e.g. weather that
   is unusual for the hour and month). A high share only means less variation is left, so a wider
   confidence interval.
2. **Share explained by the other regular predictors**, after deseasonalizing (regress each X̃ on all the
   other X̃). Above 0.8 = **flagged**: if the flagged predictor's own F-test is not significant, its
   information is already carried by the others and it is dropped.
3. **Correlated pairs** (deseasonalized correlation above 0.7): if the pair is significant *together* but not
   *one at a time*, they share one piece of information and the weaker one is dropped.

In [ ]:
def term_tests(res, terms):
    """F-test per term (all of its coefficients = 0), with the model's clustered standard errors."""
    names, rows = list(res.params.index), []
    for t in terms:
        R = np.zeros((len(TERMS[t]), len(names)))
        for i, c in enumerate(TERMS[t]):
            R[i, names.index(c)] = 1
        f = res.f_test(R)
        rows.append({"term": t, "coefficients tested": len(TERMS[t]), "F": float(np.squeeze(f.fvalue)),
                     "p": float(f.pvalue)})
    return pd.DataFrame(rows).set_index("term")


single = term_tests(res_all, ALL_TERMS)
single["significant at 5%"] = np.where(single["p"] < ALPHA, "yes", "no")
single.sort_values("p")

In [ ]:
corr = Xt_tr[CONTINUOUS].corr()
corr.round(2)

In [ ]:
# Auxiliary regressions from one cross-product matrix of the deseasonalized predictors (fast, exact).
XtX = Xt_tr.to_numpy().T @ Xt_tr.to_numpy()
col = {c: i for i, c in enumerate(Xt_tr.columns)}


def explained_by_others(v):
    j = col[v]
    o = [i for i in range(len(col)) if i != j]
    b = np.linalg.solve(XtX[np.ix_(o, o)], XtX[o, j])
    return 1 - (XtX[j, j] - XtX[o, j] @ b) / XtX[j, j]


aux = pd.DataFrame({"explained by Z": seasonal_share[CONTINUOUS],
                    "explained by the other predictors (after Z)": [explained_by_others(c) for c in CONTINUOUS]},
                   index=CONTINUOUS)
aux["explained by Z and the others together"] = (
    1 - (1 - aux["explained by Z"]) * (1 - aux["explained by the other predictors (after Z)"]))
aux["flagged"] = aux["explained by the other predictors (after Z)"] > AUX_R2_FLAG
aux

In [ ]:
def with_hierarchy(terms):
    """Remove a combined term if a term it is built from is missing."""
    terms = list(terms)
    while any(t in PARENTS and not set(PARENTS[t]) <= set(terms) for t in terms):
        terms = [t for t in terms if not (t in PARENTS and not set(PARENTS[t]) <= set(terms))]
    return terms


def joint_p(res, columns):
    """p-value of the F-test that all of these coefficients are zero (clustered by day)."""
    names = list(res.params.index)
    R = np.zeros((len(columns), len(names)))
    for i, c in enumerate(columns):
        R[i, names.index(c)] = 1
    return float(res.f_test(R).pvalue)


drop, decisions = set(), []
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
for (a, b), r in pairs[pairs.abs() > PAIR_CORR_FLAG].items():
    joint, pa, pb = joint_p(res_all, [a, b]), single.loc[a, "p"], single.loc[b, "p"]
    collinear = joint < ALPHA and max(pa, pb) >= ALPHA
    weaker = a if pa >= pb else b
    if collinear:
        drop.add(weaker)
    decisions.append({"check": f"pair {a} / {b} (correlation {r:.2f})", "p together": f"{joint:.4f}",
                      "p one at a time": f"{pa:.4f} / {pb:.4f}",
                      "decision": f"drop {weaker}" if collinear else "keep both"})
for v in aux.index[aux["flagged"]]:
    p = single.loc[v, "p"]
    if p >= ALPHA:
        drop.add(v)
    decisions.append({"check": f"flagged {v} ({aux.loc[v, 'explained by the other predictors (after Z)']:.2f} "
                               f"explained by the others)", "p one at a time": f"{p:.4f}",
                      "decision": f"drop {v}" if p >= ALPHA else "keep (adds information of its own)"})

CANDIDATES = with_hierarchy([t for t in ALL_TERMS if t not in drop])
print("dropped for collinearity:", sorted(set(ALL_TERMS) - set(CANDIDATES)) or "none")
print("candidate terms for model selection:", ", ".join(CANDIDATES))
checks = pd.DataFrame(decisions, columns=["check", "p together", "p one at a time", "decision"]).set_index("check")
checks.fillna("") if decisions else print("no pair above the correlation cut-off and no flagged predictor: nothing to test")

**Results.**

- **Significant:** both lags, zone, zone × weekend, rain (all p < 0.001) and sunlight (p = 0.03).
- **Not significant, kept as candidates:** `is_day` (0.14), cloud cover (0.16), temperature (0.54), temperature² (0.60),
  snow on the ground (0.89) and snowfall (0.91).
- **Collinearity: nothing is flagged.** After deseasonalizing, no predictor is more than 80% explained by the others.
  The most overlapping are the two lags (64% and 67%, mostly through the zone level). Their correlation is 0.78, so they
  were tested together and one at a time: each is significant on its own, so **both are kept**.
- **Change from 06:** 06 counted the calendar in the same measure ("explained by Z and the others together") and dropped
  temperature (87% there) and `is_day` (82%). That overlap is seasonality, which FWL has already removed, so they stay
  in as candidates here.

## 8. Model selection: stepwise by BIC

Z is always in the model (it is the deseasonalization), so stepwise only chooses among the regular
predictors left after the collinearity checks. **BIC** rewards a better fit and penalises every extra
parameter. Stepwise builds models one term at a time:
- **Forward:** starts from Z only (S0) and adds, at each step, the term that lowers BIC the most (S1, S2,
  ...). It stops when no term lowers BIC. A term can also be removed if that lowers BIC.
- **Backward:** starts from the full model (every candidate) and removes terms while BIC improves.

**The model stepwise ends at is the chosen model.** The models are compared here on the training data
(adjusted R², AIC, BIC); the next section only *evaluates* them on the test days. (06 picked the model with
the lowest test RMSE instead. Choosing on the test days makes their error look better than it really is,
and here it would pick on differences of a few hundredths of a pickup.)

In [ ]:
# Every model's fit comes from the same cross-product matrices (exact OLS, no refitting of the big matrix).
Xy = Xt_tr.to_numpy().T @ Yt_tr["y"].to_numpy()
yy = float(Yt_tr["y"] @ Yt_tr["y"])


def solve(terms):
    """OLS of Ỹ on the X̃ of these terms: (sum of squared residuals, coefficients, column positions)."""
    S = [col[c] for t in terms for c in TERMS[t]]
    if not S:
        return yy, np.array([]), S
    b = np.linalg.solve(XtX[np.ix_(S, S)], Xy[S])
    return yy - Xy[S] @ b, b, S


def train_stats(terms):
    ssr, b, S = solve(terms)
    return fit_stats(ssr, len(S))


def bic(terms):
    return train_stats(terms)["BIC"]


def stepwise(pool, start):
    current, path = list(start), []
    best = bic(current)
    while True:
        moves = [current + [t] for t in pool if t not in current] + [[c for c in current if c != t] for t in current]
        moves = [m for m in moves if with_hierarchy(m) == m]
        if not moves:
            break
        nxt = min(moves, key=bic)
        if bic(nxt) >= best:
            break
        added, removed = set(nxt) - set(current), set(current) - set(nxt)
        change = f"+ {added.pop()}" if added else f"- {removed.pop()}"
        current, best = nxt, bic(nxt)
        path.append({"change": change, "terms": list(current), "BIC": best})
    return path


forward = stepwise(CANDIDATES, [])
backward = stepwise(CANDIDATES, CANDIDATES)
for name, path in [("forward", forward), ("backward", backward)]:
    print(f"{name}: {len(path)} steps")
    for i, p in enumerate(path, 1):
        print(f"  {i:>2}. {p['change']:<24} BIC = {p['BIC']:,.0f}")

MODELS = {"S0: Z only (seasonal pattern)": []}
MODELS |= {f"S{i}: {p['change']}": p["terms"] for i, p in enumerate(forward, 1)}
MODELS["Full model (all candidates)"] = CANDIDATES
if backward:
    MODELS["Backward stepwise"] = backward[-1]["terms"]
names = {}                                   # one row per distinct model; duplicates joined with " = "
for name, terms in MODELS.items():
    names.setdefault(frozenset(terms), []).append(name)
MODELS = {" = ".join(labels): MODELS[labels[0]] for labels in names.values()}

# The chosen model: where stepwise ends (forward or backward, whichever has the lower BIC).
ends = [path[-1] for path in (forward, backward) if path]
pick = min(ends, key=lambda p: p["BIC"])["terms"] if ends else []
CHOSEN = next(name for name, terms in MODELS.items() if set(terms) == set(pick))
print(f"\nstepwise picks: {CHOSEN}")

selection = pd.DataFrame({name: train_stats(terms) for name, terms in MODELS.items()}).T
selection["parameters"] = selection["parameters"].astype(int)
selection

In [ ]:
steps = selection.loc[selection.index.str.match(r"S\d")]
labels = steps.index.str.split(": ").str[1].str.split(" = ").str[0]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharex=True)
for ax, c, title in [(axes[0], "R2_adj", "Adjusted R² (train)"), (axes[1], "BIC", "BIC (train, lower is better)")]:
    ax.plot(range(len(steps)), steps[c], marker="o", ms=5, color=BLUE)
    ax.set_title(title)
    ax.set_xticks(range(len(steps)), labels, rotation=60, ha="right", fontsize=8)
fig.suptitle("Forward stepwise: each point adds one term", x=0.01, ha="left", fontsize=12, fontweight="bold")
fig.tight_layout()
plt.show()

**Results.**

- **Forward** adds, in this order: last week's demand, yesterday's demand, zone, zone × weekend, rain, sunlight, cloud
  cover, `is_day`, temperature, temperature². That gives 10 models, **S1 to S10**. It stops there: snowfall and snow on
  the ground don't lower BIC.
- **Backward** starts from the full model and removes snowfall and snow on the ground, ending at the same model as S10.
- **Stepwise picks S10** (171 parameters, BIC 313,632; the full model has 313,654).
- **What does the work:** the two lags take adjusted R² from 0.56 (Z only) to 0.91. From S7 on, every step changes
  adjusted R² only in the fifth decimal.
- **The professor's point about chasing BIC shows here:** BIC keeps cloud cover, `is_day` and temperature, which are not
  significant in the F-tests. BIC treats all 345,000 rows as independent, so tiny improvements count, while the F-tests
  correctly treat hours on the same day as related.

## 9. Forecast accuracy on the test days

Predictions for the test days (never used in fitting, including in the FWL step): the seasonal part from Z
plus the regular predictors' part, converted back from `log(pickups + 1)` to pickups.

| Measure | Meaning | Better |
|---|---|---|
| RMSE | typical prediction error in pickups per zone-hour; big misses count more | lower |
| MAE | average prediction error in pickups per zone-hour | lower |

Two baselines: **calendar and zone only** (Z + zone + zone × weekend, no weather and no lags), and **same hour last week** (predict each zone-hour with last week's pickups). A useful model must beat both.

In [ ]:
def predict(terms):
    """Pickups predicted for the test days: seasonal part (Z) + regular predictors, back from logs."""
    _, b, S = solve(terms)
    log_pred = Yseas_te["y"].to_numpy() + (Xt_te.to_numpy()[:, S] @ b if S else 0)
    return pd.Series(np.expm1(log_pred).clip(min=0), index=test.index)


def accuracy(pred, rows=None):
    rows = test.index if rows is None else rows
    return {"RMSE_test": rmse(pred[rows], test.loc[rows, "pickups"]),
            "MAE_test": meanabs(pred[rows], test.loc[rows, "pickups"])}


CALENDAR_ZONE = [t for t in ["zone", "zone_x_weekend"] if t in CANDIDATES]
preds = {name: predict(terms) for name, terms in MODELS.items()}
forecast = pd.DataFrame({name: {"parameters": K_Z + sum(len(TERMS[t]) for t in MODELS[name]), **accuracy(p)}
                         for name, p in preds.items()}).T
baseline = predict(CALENDAR_ZONE)
forecast.loc["Baseline: calendar and zone only, no weather"] = pd.Series(
    {"parameters": K_Z + sum(len(TERMS[t]) for t in CALENDAR_ZONE), **accuracy(baseline)})
has_lw = test.index[test["lag168"].notna()]
LAST_WEEK = "Baseline: same hour last week" if "log_lag168" in CONTINUOUS else "Reference: same hour last week (not a model input)"
forecast.loc[LAST_WEEK] = pd.Series({"parameters": np.nan, **accuracy(test["lag168"], has_lw)})
forecast["parameters"] = forecast["parameters"].astype("Int64")

print(f"chosen in the model-selection section: {CHOSEN}")
forecast.join(selection[["R2_adj", "BIC"]])

In [ ]:
steps = forecast.loc[forecast.index.str.match(r"S\d")]
labels = steps.index.str.split(": ").str[1].str.split(" = ").str[0]
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(range(len(steps)), steps["RMSE_test"], marker="o", ms=5, color=BLUE)
ax.set_xticks(range(len(steps)), labels, rotation=60, ha="right", fontsize=8)
for name, color in [(LAST_WEEK, INK_2), ("Baseline: calendar and zone only, no weather", AQUA)]:
    ax.axhline(forecast.loc[name, "RMSE_test"], color=color, lw=1.2, ls="--")
    ax.annotate(name.split(": ")[1], (0, forecast.loc[name, "RMSE_test"]), xytext=(4, 4),
                textcoords="offset points", color=color, fontsize=9)
ax.set(title="RMSE on the test days (pickups per zone-hour), forward stepwise models")
fig.tight_layout()
plt.show()

In [ ]:
test["model"] = preds[CHOSEN]
test["baseline"] = baseline
by_zone = test.groupby("zone").apply(lambda g: pd.Series({
    "mean pickups": g["pickups"].mean(), "RMSE model": rmse(g["model"], g["pickups"]),
    "RMSE calendar and zone only": rmse(g["baseline"], g["pickups"]),
    "RMSE last week": rmse(g["lag168"].dropna(), g.loc[g["lag168"].notna(), "pickups"])}), include_groups=False)
print(f"chosen model beats calendar and zone only in {(by_zone['RMSE model'] < by_zone['RMSE calendar and zone only']).sum()} "
      f"of the {len(by_zone)} zones, and same hour last week in {(by_zone['RMSE model'] < by_zone['RMSE last week']).sum()}")
by_zone.sort_values("mean pickups", ascending=False).head(15)

In [ ]:
zone = by_zone["mean pickups"].idxmax()
june = test[(test["zone"] == zone) & (test["hour"].dt.month == 6)].set_index("hour")
june = june.reindex(pd.date_range("2025-06-01", "2025-06-30 23:00", freq="h"))   # gaps = training days
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(june.index, june["pickups"], color=BLUE, label="actual", lw=2)
ax.plot(june.index, june["model"], color=ORANGE, label="chosen model", lw=1.6)
ax.plot(june.index, june["baseline"], color=AQUA, label="calendar and zone only", lw=1.6)
ax.set(title=f"Zone {zone}, June test days (gaps are training days): hourly pickups", ylabel="pickups per hour")
ax.legend(frameon=False, ncols=3, loc="upper left")
fig.tight_layout()
plt.show()

**Results.**

- **The chosen model (S10)** has RMSE **33.99** and MAE **19.15** pickups per zone-hour on the test days. Same hour
  last week has 40.12 and 22.00, so the model's typical error is **15% smaller**. Calendar and zone alone (no weather, no
  lags) is far worse: 60.12.
- **The lags do almost all of the work.** S2 (Z + the two lags only) already has RMSE 33.99. Everything added after it
  changes the test error by less than 0.7 pickups, and the full model (33.98) is practically the same as S10.
- **By zone:** S10 beats same hour last week in 49 of the 50 zones, and calendar and zone only in all 50.
- **Holidays in the test days:** Feb 17, Jul 4 and Nov 27 (Jan 1 drops out: it has no last-week value).
- **June plot, zone 237:** the model follows the daily cycle and most of the midday peaks; calendar and zone only stays
  flat at about 200 to 350 pickups.

## Notes for the report

- **The model is used one day ahead**, since it needs yesterday's demand. For a 2026 projection, yesterday and last
  week aren't known, which is the professor's objection: compare with `07b` and `07c`, which have no lags.
- **FWL removes the dummies from the model, not their parameters:** Z's 65 coefficients are estimated in the first step
  and counted in adjusted R², AIC and BIC.
- **Autocorrelation** (weather persists over hours) is handled in the inference with standard errors clustered by day.
- **Demand = completed pickups.** People who couldn't find a taxi aren't counted, so busy hours are probably
  underestimated.